# Lab | Transformers

---

### Section structure

1. The open-source ecosystem: increasing accessibility to machine learning (ML) software and hardware
2. Some simple code demonstrations
3. Q&A

## 1. Ease-of-use: Using Transformers in 3 lines of code


**Overview of different tasks that can be automated with ML**
* Key ingredients: (1) a model trained on a specific task; (2) input data (e.g. texts or images); (3) output produced by the model.
* Transformers are currently the most popular type of deep learning algorithm. Most tasks below are solved with Transformers. There might be other types of algorithms coming up in the medium term.



**Install the Transformers library & dependencies**

In [1]:
!pip install transformers  # The Transformers library from Hugging Face
!pip install sentencepiece
!pip install wikipedia
!pip install accelerate
!pip install tf-keras
!pip install torch
!pip install transformers sentencepiece accelerate --upgrade

# NOTE: you might need to restart you jupyter kernel after installing the libraries

**The Hugging Face Pipeline**
* Makes automation of many NLP tasks possible in 3 lines of code
* Detailed documentation is available [here](https://huggingface.co/transformers/main_classes/pipelines.html)

In [2]:
from transformers import pipeline
import pandas as pd
import numpy as np
from pprint import pprint

Note : You might need more libraries or updates to run the cells below, if that is the case, follow the error messages and pip install accordingly. Chat gpt can help you if given the error messages.

### 2.1 Many models tailored to specific tasks


#### 2.1.1 Text classification

Let's select a popular text classification model in the [HF model hub](https://huggingface.co/models?pipeline_tag=text-classification&sort=downloads).

Here we chose "cardiffnlp/twitter-roberta-base-irony".

We will classify text into ironic or non ironic.

In [3]:
pipeline_classification = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-irony")  # cardiffnlp/twitter-roberta-base-irony, SamLowe/roberta-base-go_emotions

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Now that we have the model we can pass it a string and have it give us a classification.

Feel free to experiment with different sentences by changing the contents of the variable text

In [4]:
text = "Well that workshop was totally worth my time..."  # "Well that workshop was totally worth my time..."  "This smells weird, I'm not sure if I should eat this ... Yikes, it tasted like old socks!"
output = pipeline_classification(text, top_k=10)
print(output)

[{'label': 'irony', 'score': 0.9424386620521545}, {'label': 'non_irony', 'score': 0.057561349123716354}]


In [5]:
text = "This smells weird, I'm not sure if I should eat this ..."  # "Well that workshop was totally worth my time..."  "This smells weird, I'm not sure if I should eat this ... Yikes, it tasted like old socks!"
output = pipeline_classification(text, top_k=10)
print(output)

[{'label': 'non_irony', 'score': 0.9739718437194824}, {'label': 'irony', 'score': 0.02602819725871086}]


In [6]:
text = "Well isn't this a nice surprise..."
output = pipeline_classification(text, top_k=10)
print(output)

[{'label': 'irony', 'score': 0.955659806728363}, {'label': 'non_irony', 'score': 0.044340234249830246}]


Let's make the output a little cleaner

In [7]:
# make output a bit cleaner
df_output = pd.DataFrame(output)
print(df_output)

       label    score
0      irony  0.95566
1  non_irony  0.04434


As you can see, in a few lines of code and by leveraging an existing model we can classify text as ironic or non ironic. Now you have one more tool in your machine learning toolbox.

Remember that : 'when you only have a hammer everything is a nail'. But if we want to build a house (perform machine learning the right way), we need to use the right tool for the right job.

#### 2.1.2 Machine Translation

* Open source machine translation (MT) models enable you to translate between many different languages without Google Translate.
* [University of Helsinki](https://huggingface.co/Helsinki-NLP) uploaded models for more than 1000 language pairs to the Hugging Face hub
* [Facebook AI](https://huggingface.co/models?search=facebook+m2m) open-sourced several multi-lingual models
* The [EasyNMT library](https://github.com/UKPLab/EasyNMT), provides an easy wrapper for all these models
* Most machine translation models translate between two languages in one direction (e.g. German to English, but not English to German), some can translate in multiple directions.


In [10]:
# translation pipeline docs: https://huggingface.co/transformers/main_classes/pipelines.html#transformers.TranslationPipeline
#pipeline_translate = pipeline("translation", model="facebook/m2m100_418M") KeyError
# pipeline_translate = pipeline("text2text-generation", model="facebook/m2m100_418M") KeyError
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

model_name = "facebook/m2m100_418M"

tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

def pipeline_translate(text, src_lang="de", tgt_lang="en"):
    tokenizer.src_lang = src_lang

    encoded = tokenizer(text, return_tensors="pt")

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang)
    )

    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    return [{"translation_text": translation}]

tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

M2M100 is a multilingual encoder-decoder (seq-to-seq) model trained for Many-to-Many multilingual translation.

The model that can directly translate between the 9,900 directions of 100 languages.

Here we specify to translate from German 'de' to English 'en'

In [11]:
text = "Ich bin ein Fisch"
pipeline_translate(text, src_lang="de", tgt_lang="en")

[{'translation_text': 'I am a fish'}]

Let's do the same but with and entire wikipedia page in german.

In [14]:
# download any text from wikipedia, via  https://pypi.org/project/wikipedia/
import wikipedia
wikipedia.set_lang("de")

text = """
Donald Trump ist ein US-amerikanischer Politiker, Unternehmer und Medienpersönlichkeit.
Er war von 2017 bis 2021 der 45. Präsident der Vereinigten Staaten.
"""

text = text.replace('\n', ' ')[:318]
print(f"Original text:\n{text}\n")

# translate the text from wikipedia
text_translated = pipeline_translate(text, src_lang="de", tgt_lang="en")
print(f"Translated text:\n{text_translated[0]['translation_text']}")


Original text:
 Donald Trump ist ein US-amerikanischer Politiker, Unternehmer und Medienpersönlichkeit. Er war von 2017 bis 2021 der 45. Präsident der Vereinigten Staaten. 

Translated text:
Donald Trump is an American politician, entrepreneur and media personality. he was the 45th President of the United States from 2017 to 2021.


#### 2.1.3 Text Summarization

In [16]:
# docs for summarisation pipeline: https://huggingface.co/transformers/main_classes/pipelines.html#summarizationpipeline
# pipeline_summarize = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")  # sshleifer/distilbart-cnn-12-6 , google/pegasus-cnn_dailymail
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "sshleifer/distilbart-cnn-12-6"

tokenizer_sum = AutoTokenizer.from_pretrained(model_name)
model_sum = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def pipeline_summarize(text, min_length=5, max_length=30):

    inputs = tokenizer_sum(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model_sum.generate(
        inputs["input_ids"],
        min_length=min_length,
        max_length=max_length,
        num_beams=4
    )

    summary = tokenizer_sum.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return [{"summary_text": summary}]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

In [17]:
# download any long text from wikipedia, via  https://pypi.org/project/wikipedia/
import wikipedia
wikipedia.set_lang("en")

text_long = """
Rumen Georgiev Radev[a] (born 18 June 1963) is a Bulgarian politician and former Bulgarian Air Force officer who is the prime minister of Bulgaria. He previously served as president of Bulgaria from 2017 until his resignation in 2026, becoming the first head of state to resign in Bulgaria's post-Communist history.

Born in Dimitrovgrad, Radev served as commander of the Bulgarian Air Force before joining politics, holding the rank of major general. He won the 2016 Bulgarian presidential election as an independent candidate supported by the Bulgarian Socialist Party (BSP), defeating GERB candidate Tsetska Tsacheva in the runoff. The first term of Radev's presidency often saw him in conflict with then prime minister Boyko Borisov of GERB. He secured a second term in the 2021 Bulgarian general election, with 66% of the vote in the runoff.

During his second term, Radev presided over the five-year Bulgarian political crisis. Following his resignation in January 2026, he established the Progressive Bulgaria (PB) party to contest the 2026 Bulgarian parliamentary election. PB went on to receive 44.6% of the vote and an absolute majority of seats, effectively ending the political crisis. This made Radev the first person in Bulgaria's post-Communist history to serve as both President and Prime Minister.


"""
text_long = text_long.replace('\n', ' ')
print(f"Original text:\n{text_long}\n")

# translate the text from wikipedia
text_summarized = pipeline_summarize(text_long, min_length=5, max_length=30)
print(f"Summarized text:\n{text_summarized[0]['summary_text']}")

Original text:
 Rumen Georgiev Radev[a] (born 18 June 1963) is a Bulgarian politician and former Bulgarian Air Force officer who is the prime minister of Bulgaria. He previously served as president of Bulgaria from 2017 until his resignation in 2026, becoming the first head of state to resign in Bulgaria's post-Communist history.  Born in Dimitrovgrad, Radev served as commander of the Bulgarian Air Force before joining politics, holding the rank of major general. He won the 2016 Bulgarian presidential election as an independent candidate supported by the Bulgarian Socialist Party (BSP), defeating GERB candidate Tsetska Tsacheva in the runoff. The first term of Radev's presidency often saw him in conflict with then prime minister Boyko Borisov of GERB. He secured a second term in the 2021 Bulgarian general election, with 66% of the vote in the runoff.  During his second term, Radev presided over the five-year Bulgarian political crisis. Following his resignation in January 2026, he esta

#### 2.1.4 Named Entity Recognition

NER is a task that involves identifying and classifying specific entities in text into predefined categories, such as names of people, organizations, locations, dates, and more.

For example, in the sentence "Apple Inc. was founded by Steve Jobs in California," NER would recognize "Apple Inc." as an organization, "Steve Jobs" as a person, and "California" as a location.

In [18]:
pipeline_ner = pipeline("token-classification", model="dslim/bert-base-NER-uncased", aggregation_strategy="simple")

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER-uncased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [19]:
import wikipedia
wikipedia.set_lang("en")

text_long = wikipedia.summary("Donald Trump").replace('\n', ' ')

output = pipeline_ner(text_long)

pd.DataFrame(output)

,entity_group,score,word,start,end
0,PER,0.984957,donald john trump,0,17
1,MISC,0.984379,american,45,53
2,LOC,0.988764,united states,134,147
3,MISC,0.822807,republican,165,175
4,ORG,0.680199,party,176,181
5,LOC,0.880766,new york city,254,267
6,PER,0.976201,trump,276,281
7,ORG,0.674295,university of pennsylvania,301,327
8,ORG,0.848119,trump organization,460,478
9,PER,0.870318,trump,607,612


### 2.2. Universal models

The models mentioned above are designed to excel at a single specific task on a particular dataset. The key advantage of these models is their high performance and accuracy on that specific task and dataset.

However, in real-world applications, the problems you'll face often require solving slightly different tasks, possibly with varied category definitions or applied to different types of texts.

Universal models can help address this challenge. Although they also focus on one task, the task is general or universal enough that many other tasks can be reformulated into it. Two examples of universal tasks are:

- Natural Language Inference (NLI): A task that can effectively solve a wide range of classification tasks by determining whether a given premise supports, contradicts, or is neutral with respect to a hypothesis.

- Token Generation: An even more universal task that can be applied to solve virtually any text-related task, including translation, summarization, and text completion.

These universal tasks enable the models to be versatile and adaptable to various problems beyond the specific ones they were initially trained on.

#### Zero-shot classification


Zero-shot classification is a technique where a model can categorize data into classes it has never seen before.

Instead of relying on labeled examples for each class, the model understands the relationship between the input and the class descriptions, allowing it to make accurate predictions without needing specific training on those classes.

In [20]:
pipeline_zeroshot_classification = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Here we will give the model a list of classes ('payment issues', 'travel advice', 'bug report') for it to classify our string.

In [22]:
#text = "Customer: I have not received my reimbursement yet. What the hell is going on?"
#classes = ['payment issues', 'travel advice', 'bug report']  # "account opening", "customer complaint"

text = "I do not think the government is trustworthy anymore. We need to mobilize and resist!"
classes = ["civil disobedience", "praise of the government", "travel advice"]  # "collective action"

output = pipeline_zeroshot_classification(text, classes, multi_label=True)

pd.DataFrame(data=[output["labels"], output["scores"]], index=["class", "probability"]).T


,class,probability
0,civil disobedience,0.784814
1,travel advice,0.167968
2,praise of the government,0.000854


## Exercise

Now it is your turn to go to the hugging face library https://huggingface.co/models?pipeline_tag=text-classification&sort=downloads

(you can select on the left menu of the website the type of NLP tasks you want models to perform)

- Find an NLP model that we have not used previously.

- Get some data from wikipedia or elsewhere.

- Perform inference with the model and print the result!

- Comment your code along the way, describe what your model does and what your end goal is from input to output.

Have fun!

In [23]:
# Goal:
# Use a Fill-Mask NLP model to predict a missing word in a sentence.
# The input is a sentence with one hidden word: [MASK]
# The output is the model's best guesses for the missing word.
# Your code here :
pipeline_fill_mask = pipeline("fill-mask", model="distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [24]:
text = "The capital of Bulgaria is [MASK]."
output = pipeline_fill_mask(text)
print(output)

[{'score': 0.9307059049606323, 'token': 8755, 'token_str': 'sofia', 'sequence': 'the capital of bulgaria is sofia.'}, {'score': 0.06330259889364243, 'token': 26307, 'token_str': 'ruse', 'sequence': 'the capital of bulgaria is ruse.'}, {'score': 0.002749301493167877, 'token': 8063, 'token_str': 'bulgaria', 'sequence': 'the capital of bulgaria is bulgaria.'}, {'score': 0.0004149205924477428, 'token': 29255, 'token_str': 'skopje', 'sequence': 'the capital of bulgaria is skopje.'}, {'score': 0.00031368155032396317, 'token': 23162, 'token_str': 'thessaloniki', 'sequence': 'the capital of bulgaria is thessaloniki.'}]


In [25]:
df_output = pd.DataFrame(output)
df_output[["token_str", "score", "sequence"]]

,token_str,score,sequence
0,sofia,0.930706,the capital of bulgaria is sofia.
1,ruse,0.063303,the capital of bulgaria is ruse.
2,bulgaria,0.002749,the capital of bulgaria is bulgaria.
3,skopje,0.000415,the capital of bulgaria is skopje.
4,thessaloniki,0.000314,the capital of bulgaria is thessaloniki.


In [26]:
text = "My mother is the most [MASK] woman in the world!"
output = pipeline_fill_mask(text)
print(output)

[{'score': 0.4618646800518036, 'token': 3376, 'token_str': 'beautiful', 'sequence': 'my mother is the most beautiful woman in the world!'}, {'score': 0.061722539365291595, 'token': 3928, 'token_str': 'powerful', 'sequence': 'my mother is the most powerful woman in the world!'}, {'score': 0.037098828703165054, 'token': 6919, 'token_str': 'wonderful', 'sequence': 'my mother is the most wonderful woman in the world!'}, {'score': 0.020248783752322197, 'token': 3297, 'token_str': 'famous', 'sequence': 'my mother is the most famous woman in the world!'}, {'score': 0.01307690143585205, 'token': 6429, 'token_str': 'amazing', 'sequence': 'my mother is the most amazing woman in the world!'}]


In [27]:
df_output = pd.DataFrame(output)
df_output[["token_str", "score", "sequence"]]

,token_str,score,sequence
0,beautiful,0.461865,my mother is the most beautiful woman in the w...
1,powerful,0.061723,my mother is the most powerful woman in the wo...
2,wonderful,0.037099,my mother is the most wonderful woman in the w...
3,famous,0.020249,my mother is the most famous woman in the world!
4,amazing,0.013077,my mother is the most amazing woman in the world!


In [28]:
text = "The largest city in Spain is [MASK]."
output = pipeline_fill_mask(text)
print(output)

[{'score': 0.26688051223754883, 'token': 6921, 'token_str': 'madrid', 'sequence': 'the largest city in spain is madrid.'}, {'score': 0.16963358223438263, 'token': 18983, 'token_str': 'seville', 'sequence': 'the largest city in spain is seville.'}, {'score': 0.12644386291503906, 'token': 25744, 'token_str': 'zaragoza', 'sequence': 'the largest city in spain is zaragoza.'}, {'score': 0.10887114703655243, 'token': 7623, 'token_str': 'barcelona', 'sequence': 'the largest city in spain is barcelona.'}, {'score': 0.06493693590164185, 'token': 27382, 'token_str': 'malaga', 'sequence': 'the largest city in spain is malaga.'}]


In [29]:
df_output = pd.DataFrame(output)
df_output[["token_str", "score", "sequence"]]

,token_str,score,sequence
0,madrid,0.266881,the largest city in spain is madrid.
1,seville,0.169634,the largest city in spain is seville.
2,zaragoza,0.126444,the largest city in spain is zaragoza.
3,barcelona,0.108871,the largest city in spain is barcelona.
4,malaga,0.064937,the largest city in spain is malaga.


In [30]:
text = "The best programming language is [MASK]."
output = pipeline_fill_mask(text)
print(output)
df_output = pd.DataFrame(output)
df_output[["token_str", "score", "sequence"]]

[{'score': 0.128677636384964, 'token': 18750, 'token_str': 'python', 'sequence': 'the best programming language is python.'}, {'score': 0.08588561415672302, 'token': 9262, 'token_str': 'java', 'sequence': 'the best programming language is java.'}, {'score': 0.04169420152902603, 'token': 25718, 'token_str': 'php', 'sequence': 'the best programming language is php.'}, {'score': 0.0236127320677042, 'token': 2394, 'token_str': 'english', 'sequence': 'the best programming language is english.'}, {'score': 0.01859373040497303, 'token': 1039, 'token_str': 'c', 'sequence': 'the best programming language is c.'}]


,token_str,score,sequence
0,python,0.128678,the best programming language is python.
1,java,0.085886,the best programming language is java.
2,php,0.041694,the best programming language is php.
3,english,0.023613,the best programming language is english.
4,c,0.018594,the best programming language is c.
